# Acronym Disambiguation — DictaBERT Cross-Encoder Training (Colab, self-contained)

Fine-tunes [DictaBERT](https://huggingface.co/dicta-il/dictabert) as a binary
cross-encoder: given a (marked sentence, candidate expansion) pair, score whether
the candidate fits. At inference, score every candidate for an item and take the
argmax.

**Runtime:** `Runtime > Change runtime type > T4 GPU` before running.

**Branch:** fetches from `ben-train`, not `main` — that branch has the updated
`train.csv` (2829 rows, after releasing previously-held rows). Switch `REPO_RAW`
in the next cell if you need a different branch.

This is a from-scratch Colab notebook: no repo clone, no package install, no
local model cache assumed. Every cell fetches what it needs over plain HTTP from
GitHub / Hugging Face, same as the original `train_dictabert.ipynb` before the
`src/hebrew_acronyms` package restructuring. It trains on the new `study_v1`
train/dev data (see `data/study_v1/encoder_inputs/`) instead of the older
`data/splits/` files.

This notebook only trains and saves a checkpoint. Evaluation (item accuracy,
error analysis) runs locally afterward against the downloaded checkpoint.

## 1. Setup

In [ ]:
!pip install -q transformers torch

In [ ]:
import urllib.request
import random, time
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm

REPO_RAW = 'https://raw.githubusercontent.com/BenCarmel123/hebrew-acronym-disambiguation/ben-train'

# Pull this run's exact preprocessing code from the repo rather than redefining it
# here, so training can never quietly diverge from the committed pairs.py.
# Note the new path: src/hebrew_acronyms/models/common/ (the old model/common/
# path no longer exists after the package restructuring).
urllib.request.urlretrieve(f'{REPO_RAW}/src/hebrew_acronyms/models/common/pairs.py', 'pairs.py')
urllib.request.urlretrieve(f'{REPO_RAW}/src/hebrew_acronyms/data_processing/common/csv_io.py', 'csv_io.py')

# pairs.py does `from hebrew_acronyms.data_processing.common.csv_io import load_rows`,
# which needs the real package path to exist. Fake just enough of the package so
# that import resolves to the csv_io.py we just fetched, without installing anything.
import sys, types
pkg = types.ModuleType('hebrew_acronyms'); pkg.__path__ = []
dp = types.ModuleType('hebrew_acronyms.data_processing'); dp.__path__ = []
common = types.ModuleType('hebrew_acronyms.data_processing.common'); common.__path__ = []
sys.modules['hebrew_acronyms'] = pkg
sys.modules['hebrew_acronyms.data_processing'] = dp
sys.modules['hebrew_acronyms.data_processing.common'] = common
import csv_io
sys.modules['hebrew_acronyms.data_processing.common.csv_io'] = csv_io

import pairs

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Load data

Fetches `train.csv` and `dev.csv` from `data/study_v1/encoder_inputs/` in the
repo — the leakage-checked, document-overlap-excluded data, with a small
natural-only (Knesset) dev set. Train is still mostly weak-labeled
(`gold_expansion` from substitution/mining, not all human-annotated); dev here
is 100% natural and human-reviewed. See `data/study_v1/encoder_inputs/manifest.json`
for exact construction/exclusion counts.

The real verdict is still the separate hand-verified `data/splits/test_items.csv`,
scored once, later, outside this notebook — untouched by any of this.

In [ ]:
def fetch_csv(path):
    local, _ = urllib.request.urlretrieve(f'{REPO_RAW}/{path}')
    return pairs.load_rows(local) if hasattr(pairs, 'load_rows') else csv_io.load_rows(local)

train_rows = fetch_csv('data/study_v1/encoder_inputs/train.csv')
dev_rows = fetch_csv('data/study_v1/encoder_inputs/dev.csv')
print('train rows:', len(train_rows))
print('dev rows:  ', len(dev_rows))

## 3. Build (marked context, candidate) pairs

One pair per candidate of every row, labeled 1 for gold and 0 otherwise. The target
acronym occurrence is wrapped in `[ACR]` / `[/ACR]` so the model knows which span is
in question. This `pairs.py` version uses the row's own explicit `span_start`/
`span_end`/`target_raw` columns (present in the study_v1 schema) rather than
re-searching the sentence for the acronym.

In [ ]:
train_pairs = pairs.build_pairs(train_rows)
dev_pairs = pairs.build_pairs(dev_rows)
print('train pairs:', len(train_pairs), '| positives:', sum(l for _, _, l in train_pairs))
print('dev pairs:  ', len(dev_pairs),   '| positives:', sum(l for _, _, l in dev_pairs))

# Sanity check: look at one built pair before trusting the pipeline on thousands more.
ctx, cand, label = train_pairs[0]
print('\nmarked context:', ctx)
print('candidate:     ', cand)
print('label:         ', label)

## 4. Model — DictaBERT + a one-unit scoring head

`[ACR]` / `[/ACR]` are added to the tokenizer as real tokens (so they get their own
trainable embedding rows instead of being split into subwords), and the embedding
matrix is resized accordingly — done **before** the optimizer is built, since
resizing replaces the embedding module.

In [ ]:
MODEL_ID = 'dicta-il/dictabert'
MAX_LEN = 256   # this corpus's sentences top out well under this

# WHICH VECTOR THE SCORING HEAD READS. The encoder produces one vector per token;
# something has to reduce those to the single vector the head scores.
#
#   'cls'        the [CLS] summary of the whole pair. The conventional default, but
#                [CLS] is a general sentence summary — nothing about it is specific to
#                the acronym being asked about.
#   'marker'     the [ACR] token's own vector. Sits exactly on the target span, so it
#                is the position the question is actually about.
#   'span_mean'  mean of the tokens BETWEEN the markers — the acronym's own subwords,
#                contextualised. Standard for span-targeted tasks.
#   'concat'     [CLS] and span_mean together (2 x hidden into the head): keeps the
#                sentence summary and adds the target signal.
#
# Change this and re-run to compare. Remember the ~5-point noise floor (three cls runs
# on the old data spanned 0.770-0.815): only a clear gap between poolings is evidence.
POOLING = 'cls'

tok = AutoTokenizer.from_pretrained(MODEL_ID)
encoder = AutoModel.from_pretrained(MODEL_ID)

n_added = tok.add_special_tokens({'additional_special_tokens': [pairs.ACR_OPEN, pairs.ACR_CLOSE]})
if n_added:
    encoder.resize_token_embeddings(len(tok))
ACR_OPEN_ID, ACR_CLOSE_ID = tok.convert_tokens_to_ids([pairs.ACR_OPEN, pairs.ACR_CLOSE])
print('markers added:', n_added, '| vocab size now:', len(tok))


class CrossEncoder(nn.Module):
    '''DictaBERT + dropout + one linear unit over the pooled vector. Emits ONE raw
    logit per pair — not a probability. Apply sigmoid for a per-pair score, or just
    compare logits across one item's candidates and take the argmax (monotonic, so
    sigmoid is not even necessary for ranking, only if you want a 0-1 number).'''

    def __init__(self, encoder, hidden, pooling='cls', dropout=0.1):
        super().__init__()
        self.encoder = encoder
        self.pooling = pooling
        self.dropout = nn.Dropout(dropout)
        # concat feeds two vectors to the head, so the head is twice as wide.
        self.score = nn.Linear(hidden * 2 if pooling == 'concat' else hidden, 1)

    def _span_mean(self, hidden_states, input_ids):
        '''Mean of the tokens strictly between [ACR] and [/ACR], per row.

        Built as a mask rather than by slicing, because the span sits at a different
        position in every row of the batch. A row whose markers were somehow lost
        falls back to [CLS] rather than dividing by zero.
        '''
        opened = (input_ids == ACR_OPEN_ID).cumsum(dim=1)
        closed = (input_ids == ACR_CLOSE_ID).cumsum(dim=1)
        # strictly inside: after the open marker, before the close marker
        inside = ((opened == 1) & (closed == 0)
                  & (input_ids != ACR_OPEN_ID)).float().unsqueeze(-1)
        counts = inside.sum(dim=1)                      # (batch, 1)
        pooled = (hidden_states * inside).sum(dim=1) / counts.clamp(min=1)
        empty = (counts.squeeze(-1) == 0)
        if empty.any():
            pooled[empty] = hidden_states[empty, 0]     # fall back to [CLS]
        return pooled

    def _marker(self, hidden_states, input_ids):
        '''The [ACR] token's own vector, per row. Falls back to [CLS] if absent.'''
        is_open = (input_ids == ACR_OPEN_ID)
        # first occurrence per row; argmax on a bool gives the first True, or 0 if none
        idx = is_open.float().argmax(dim=1)
        pooled = hidden_states[torch.arange(hidden_states.size(0)), idx]
        missing = ~is_open.any(dim=1)
        if missing.any():
            pooled[missing] = hidden_states[missing, 0]
        return pooled

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if token_type_ids is not None:
            kwargs['token_type_ids'] = token_type_ids
        h = self.encoder(**kwargs).last_hidden_state
        cls = h[:, 0]

        if self.pooling == 'cls':
            pooled = cls
        elif self.pooling == 'marker':
            pooled = self._marker(h, input_ids)
        elif self.pooling == 'span_mean':
            pooled = self._span_mean(h, input_ids)
        elif self.pooling == 'concat':
            pooled = torch.cat([cls, self._span_mean(h, input_ids)], dim=-1)
        else:
            raise ValueError(f"unknown POOLING {self.pooling!r}")

        return self.score(self.dropout(pooled)).squeeze(-1)


model = CrossEncoder(encoder, hidden=encoder.config.hidden_size,
                     pooling=POOLING).to(device)
print('pooling:', POOLING, '| parameters:', f"{sum(p.numel() for p in model.parameters()):,}")

## 5. Encoding — assemble `[CLS] context [SEP] candidate [SEP]`

Truncation matters here: HuggingFace's default pair truncation trims from the end
of the longer sequence and can silently delete `[/ACR]` on a long context. This
function truncates the context itself, from the outside in, so both markers always
survive.

In [ ]:
def encode_batch(pairs_batch, max_len=MAX_LEN):
    '''pairs_batch: list of (marked_context, candidate) -> batch dict of tensors.'''
    open_id, close_id = tok.convert_tokens_to_ids([pairs.ACR_OPEN, pairs.ACR_CLOSE])
    cls_id, sep_id = tok.cls_token_id, tok.sep_token_id
    rows = []
    for marked_context, candidate in pairs_batch:
        ctx = tok.encode(marked_context, add_special_tokens=False)
        cand = tok.encode(candidate, add_special_tokens=False)
        o, c = ctx.index(open_id), ctx.index(close_id)
        budget = max_len - 3 - len(cand)
        lo, hi = 0, len(ctx)
        while (hi - lo) > budget:
            left_room, right_room = o - lo, hi - (c + 1)
            if left_room >= right_room and left_room > 0:
                lo += 1
            elif right_room > 0:
                hi -= 1
            else:
                break
        ctx = ctx[lo:hi]
        ids = [cls_id] + ctx + [sep_id] + cand + [sep_id]
        types = [0] * (len(ctx) + 2) + [1] * (len(cand) + 1)
        rows.append((ids, types))
    width = max(len(ids) for ids, _ in rows)
    pad = tok.pad_token_id or 0
    input_ids, attention_mask, token_type_ids = [], [], []
    for ids, types in rows:
        n = width - len(ids)
        input_ids.append(ids + [pad] * n)
        attention_mask.append([1] * len(ids) + [0] * n)
        token_type_ids.append(types + [0] * n)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long, device=device),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long, device=device),
        'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long, device=device),
    }

## 6. Training loop

Binary cross-entropy per pair. Shuffled each epoch at the pair level — with the
gold vs. non-gold candidates for the same item usually landing in different
batches, which is fine for this loss (each pair is judged independently; nothing
requires an item's candidates to share a batch).

In [ ]:
BATCH_SIZE = 16
EPOCHS = 1        # 1 is what the original run selected: epochs 2 and 3 only
                  # overfit (dev loss 0.42 -> 0.53 -> 0.54 while train loss
                  # kept falling). Raise it if you lower LR or add warmup.
LR = 2e-5
SEED = 42         # Without this, batch order and dropout differ per run and
                  # two runs of the SAME config landed 3.4 points apart
                  # (0.815 vs 0.781) on the old data. Pin it, or a config change
                  # cannot be told apart from run-to-run noise.

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()

def batches(pairs, batch_size, shuffle):
    idx = list(range(len(pairs)))
    if shuffle:
        random.shuffle(idx)
    for i in range(0, len(idx), batch_size):
        chunk = [pairs[j] for j in idx[i:i + batch_size]]
        yield [(c, cand) for c, cand, _ in chunk], [l for _, _, l in chunk]

@torch.no_grad()
def evaluate(pairs):
    '''Mean BCE loss and pairwise accuracy (sigmoid > 0.5) over a pair list.
    Pairwise accuracy is NOT the same as item accuracy (picking the right
    candidate) — item accuracy is computed locally afterward, against the
    downloaded checkpoint.'''
    model.eval()
    total_loss, total_correct, n = 0.0, 0, 0
    n_batches = (len(pairs) + 31) // 32
    for ctx_batch, labels in tqdm(batches(pairs, batch_size=32, shuffle=False),
                                  total=n_batches, desc='  dev eval', leave=False):
        enc = encode_batch(ctx_batch)
        y = torch.tensor(labels, dtype=torch.float, device=device)
        logits = model(**enc)
        loss = loss_fn(logits, y)
        total_loss += loss.item() * len(labels)
        total_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
        n += len(labels)
    return total_loss / n, total_correct / n

best_dev_loss = float('inf')
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    running_loss, n_seen = 0.0, 0
    n_batches = (len(train_pairs) + BATCH_SIZE - 1) // BATCH_SIZE
    progress = tqdm(batches(train_pairs, BATCH_SIZE, shuffle=True), total=n_batches,
                    desc=f'epoch {epoch}/{EPOCHS}', unit='batch')
    for ctx_batch, labels in progress:
        enc = encode_batch(ctx_batch)
        y = torch.tensor(labels, dtype=torch.float, device=device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(**enc)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(labels)
        n_seen += len(labels)
        progress.set_postfix(loss=f'{running_loss / n_seen:.4f}')

    train_loss = running_loss / n_seen
    dev_loss, dev_pair_acc = evaluate(dev_pairs)
    print(f'epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f}  '
          f'dev_loss={dev_loss:.4f}  dev_pair_acc={dev_pair_acc:.3f}  '
          f'({time.time()-t0:.0f}s)')

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        # Saved in fp16: halves the checkpoint's file size (~700MB -> ~350MB)
        # for a faster files.download() below. This only affects the SAVED
        # artifact, not training itself (which stays fp32 throughout) -- cast
        # back to float() when loading for local CPU/MPS eval, since fp16
        # matmul isn't well supported off-GPU.
        half_state = {k: v.half() for k, v in model.state_dict().items()}
        torch.save(half_state, 'best.pt')
        print('  -> saved best.pt (fp16)')

## 7. Save checkpoint

In [ ]:
from google.colab import files
files.download('best.pt')